In [ ]:
!pip install vllm --quiet

In [ ]:
import gc
import torch
import numpy as np
import polars as pl
from pathlib import Path
from pydantic import BaseModel, Field
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/cmpe-492/data")
CONTEXT_DIR = DATA_DIR / "context"
RESULTS_DIR = DATA_DIR / "results"
TABULAR_DIR = DATA_DIR / "tabular"

In [ ]:
FEATURES = [
    "age", "gender", "side", "stone_localization", "stone_burden_cm2",
    "ct_scan", "comorbidity", "previous_surgery", "stone_anamnesis",
    "asa_score", "urine_culture", "creatinine", "solitary_kidney",
    "renal_anomaly", "additional_renal_disease",
]

COLUMN_MAP = {
    "YAŞ": "age",
    "CİNSİYET": "gender",
    "TARAF": "side",
    "LOKALİZASYON": "stone_localization",
    "TOPLAM TAŞ YÜKÜ (CM2)": "stone_burden_cm2",
    "BT": "ct_scan",
    "ÖZGEÇMİŞ": "comorbidity",
    "GEÇİRİLMİŞ CERRAHİ": "previous_surgery",
    "TAS ANAMNEZI": "stone_anamnesis",
    "ASA SKORU": "asa_score",
    "İKAB": "urine_culture",
    "KREATİNİN": "creatinine",
    "SOLİTER BB": "solitary_kidney",
    "RENAL ANOMALİ": "renal_anomaly",
    "EK RENAL HASTALIK": "additional_renal_disease",
    "SONUÇ-2": "result",
}

NUMERIC_FEATURES = ["age", "stone_burden_cm2", "creatinine"]
CATEGORICAL_FEATURES = [
    "gender", "side", "stone_localization", "ct_scan",
    "comorbidity", "previous_surgery", "stone_anamnesis",
    "asa_score", "urine_culture", "solitary_kidney",
    "renal_anomaly", "additional_renal_disease",
]

In [ ]:
class PredictionOutput(BaseModel):
    prediction: int = Field(description="1 for success (stone-free), 2 for failure (residual fragments)")
    reasoning: str = Field(description="A single short sentence explaining the prediction")
    confidence: str = Field(description="Model confidence: 'low', 'medium', or 'high'")

In [ ]:
df = pl.read_csv(TABULAR_DIR / "pediatric-pcnl.csv", infer_schema_length=None)
df = df.rename(COLUMN_MAP).select(FEATURES + ["result"])
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
display(df.head(3))


In [ ]:
feature_dict = (CONTEXT_DIR / "feature_dictionary.md").read_text()
clinical_evidence = (CONTEXT_DIR / "clinical_evidence.md").read_text()

In [ ]:
def to_float_array(s: pl.Series) -> np.ndarray:
    arr = np.array(s.cast(pl.Utf8).str.replace(",", ".").to_list(), dtype=object)
    result = np.full(len(arr), np.nan, dtype=np.float64)
    for i, v in enumerate(arr):
        try:
            result[i] = float(v)
        except (ValueError, TypeError):
            pass
    return result


vals_per_feat = {f: to_float_array(df[f]) for f in NUMERIC_FEATURES}
y = df["result"].to_numpy()
n = len(df)

class LOOStats:
    def __init__(self):
        self.class_means = {}
        self.class_stds = {}
        for feat in NUMERIC_FEATURES:
            vals = vals_per_feat[feat]
            for cls in [1, 2]:
                mask = (y == cls) & ~np.isnan(vals)
                self.class_means[(feat, cls)] = vals[mask].mean()
                self.class_stds[(feat, cls)] = vals[mask].std()

        self.cat_distributions = {}
        for feat in CATEGORICAL_FEATURES:
            vals = df[feat].to_numpy()
            overall = {}
            by_class = {1: {}, 2: {}}
            for i in range(n):
                if vals[i] is None or (isinstance(vals[i], float) and np.isnan(vals[i])):
                    continue
                key = str(vals[i])
                cls = int(y[i])
                overall[key] = overall.get(key, 0) + 1
                by_class[cls][key] = by_class[cls].get(key, 0) + 1
            total = sum(1 for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v)))
            self.cat_distributions[feat] = (overall, by_class, total)

    def get_stats(self, idx: int) -> str:
        lines = []
        for feat in NUMERIC_FEATURES:
            vals = vals_per_feat[feat]
            lines.append(f"\n{feat}:")
            for cls in [1, 2]:
                mask = (y == cls) & ~np.isnan(vals)
                count = mask.sum()
                total = vals[mask].sum()
                if mask[idx]:
                    total -= vals[idx]
                    count -= 1
                mean = total / count if count > 0 else float("nan")
                lines.append(f"  class {cls} (n={count}): mean={mean:.2f}")

        for feat in CATEGORICAL_FEATURES:
            overall, by_class, total = self.cat_distributions[feat]
            cur_val = df[feat][idx]
            lines.append(f"\n{feat}:")
            all_keys = sorted(overall.keys(), key=lambda x: overall[x], reverse=True)
            for key in all_keys:
                cls1_count = by_class[1].get(key, 0)
                cls2_count = by_class[2].get(key, 0)
                total_count = overall[key]
                if str(cur_val) == key:
                    total_count -= 1
                    if y[idx] == 1:
                        cls1_count -= 1
                    else:
                        cls2_count -= 1
                cls1_pct = cls1_count / (total - 1) * 100 if total > 1 else 0
                cls2_pct = cls2_count / (total - 1) * 100 if total > 1 else 0
                overall_pct = total_count / (total - 1) * 100 if total > 1 else 0
                lines.append(f"  '{key}': class1={cls1_pct:.1f}%, class2={cls2_pct:.1f}%, overall={overall_pct:.1f}%")
        return "\n".join(lines)


loo_stats = LOOStats()

In [ ]:
SYSTEM_PROMPT_MINIMAL = f"""You are an expert urologist predicting PCNL (percutaneous nephrolithotomy) outcomes in pediatric patients.

## Feature Reference
{feature_dict}"""

SYSTEM_PROMPT_FULL = f"""You are an expert urologist predicting PCNL (percutaneous nephrolithotomy) outcomes in pediatric patients.

## Feature Reference
{feature_dict}

## Clinical Evidence
{clinical_evidence}"""

In [ ]:
def build_user_prompt(row: dict, loo_stats_str: str) -> str:
    features = ", ".join(f"{k}: {v}" for k, v in row.items() if k != "result")
    return (
        f"## Current Patient\n{features}\n\n"
        f"## Population Statistics (excluding current patient)\n{loo_stats_str}\n\n"
        f"Predict: 1=stone-free (success), 2=residual fragments (failure).\nReply JSON only."
    )



In [ ]:
MODEL_CONFIGS = [
    {"model_id": "meta-llama/Llama-3.1-8B-Instruct", "thinking_modes": [False]},
    {"model_id": "Qwen/Qwen3.5-9B-Instruct", "thinking_modes": [False, True]},
    {"model_id": "Qwen/Qwen3.5-35B-A3B", "thinking_modes": [False, True]},
]

CONTEXTS = {
    "minimal": SYSTEM_PROMPT_MINIMAL,
    "full": SYSTEM_PROMPT_FULL,
}


def build_messages_list(system_prompt: str) -> list:
    return [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": build_user_prompt(df.row(i, named=True), loo_stats.get_stats(i))},
        ]
        for i in range(n)
    ]


messages_minimal = build_messages_list(SYSTEM_PROMPT_MINIMAL)
messages_full = build_messages_list(SYSTEM_PROMPT_FULL)

guided_sampling = SamplingParams(
    temperature=0.1,
    top_p=0.9,
    max_tokens=1024,
    structured_outputs=StructuredOutputsParams(json=PredictionOutput.model_json_schema()),
)

all_metrics = []


In [ ]:
for cfg in MODEL_CONFIGS:
    model_id = cfg["model_id"]
    model_name = model_id.split("/")[-1].lower()
    thinking_modes = cfg["thinking_modes"]

    print(f"\n{'='*60}")
    print(f"Loading: {model_id}")
    print(f"{'='*60}")

    llm = LLM(
        model=model_id,
        max_model_len=8192,
        gpu_memory_utilization=0.95,
        structured_outputs_config={"backend": "xgrammar"},
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    for context_name, system_prompt in CONTEXTS.items():
        messages = messages_minimal if context_name == "minimal" else messages_full

        for enable_thinking in thinking_modes:
            thinking_label = "on" if enable_thinking else "off"
            run_name = f"{model_name}_thinking={thinking_label}_{context_name}"
            print(f"\n{'─'*40}")
            print(f"Running: {run_name}")

            texts = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
            )

            outputs = llm.generate(texts, guided_sampling)

            results = []
            for i, output in enumerate(outputs):
                text = output.outputs[0].text.strip()
                try:
                    parsed = PredictionOutput.model_validate_json(text)
                    results.append({
                        "idx": i,
                        "actual": int(y[i]),
                        "predicted": parsed.prediction,
                        "reasoning": parsed.reasoning,
                        "confidence": parsed.confidence,
                    })
                except Exception:
                    results.append({
                        "idx": i,
                        "actual": int(y[i]),
                        "predicted": None,
                        "reasoning": None,
                        "confidence": None,
                    })

            results_df = pl.DataFrame(results)
            results_df.write_csv(RESULTS_DIR / f"predictions_{run_name}.csv")

            valid = results_df.filter(pl.col("predicted").is_not_null())
            y_true = valid["actual"].to_list()
            y_pred = valid["predicted"].to_list()

            cm = confusion_matrix(y_true, y_pred)
            report = classification_report(y_true, y_pred, target_names=["1=Success", "2=Failure"], zero_division=0, output_dict=True)

            metrics = {
                "model": model_name,
                "thinking": thinking_label,
                "context": context_name,
                "total": len(results_df),
                "valid": len(valid),
                "failed": len(results_df) - len(valid),
                "accuracy": accuracy_score(y_true, y_pred),
                "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
                "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
                "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
                "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
                "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
                "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
                "success_precision": report["1=Success"]["precision"],
                "success_recall": report["1=Success"]["recall"],
                "success_f1": report["1=Success"]["f1-score"],
                "success_support": report["1=Success"]["support"],
                "failure_precision": report["2=Failure"]["precision"],
                "failure_recall": report["2=Failure"]["recall"],
                "failure_f1": report["2=Failure"]["f1-score"],
                "failure_support": report["2=Failure"]["support"],
                "tn": int(cm[0][0]),
                "fp": int(cm[0][1]),
                "fn": int(cm[1][0]),
                "tp": int(cm[1][1]),
            }

            all_metrics.append(metrics)

            print(f"  Accuracy: {metrics['accuracy']:.4f}")
            print(f"  F1 (weighted): {metrics['f1_weighted']:.4f}")
            print(f"  F1 (macro): {metrics['f1_macro']:.4f}")
            print(f"  Confusion matrix: TP={metrics['tp']} FP={metrics['fp']} FN={metrics['fn']} TN={metrics['tn']}")

    del llm
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\nGPU memory cleared after {model_name}")



In [ ]:
metrics_df = pl.DataFrame(all_metrics)
metrics_df.write_csv(RESULTS_DIR / "experiment_summary.csv")
display(metrics_df)



